# RGB - 04 Test externo: Unicamp-UAV

Que hace: mide un modelo ya entrenado como **detector de personas** sobre el test de Unicamp-UAV (4K, dataset que el modelo nunca vio).
Metricas del sistema, no de entrenamiento: P / R / F1 a umbrales de operacion, cruce con IoU 0.5, sobre la imagen completa.

Que NO hace: no entrena ni toca el dataset de VisDrone.

## Parametros

`UNICAMP_ZIP` es el zip del repo de Unicamp-UAV que subiste a Drive. `LIMIT = 50` para una prueba rapida; `None` para todo el test.

In [ ]:
UNICAMP_ZIP = "MyDrive/deteccion_drones/datasets/unicamp-test/Unicamp-UAV-Dataset-main.zip"      # ruta del zip en Drive
UNICAMP_DIR = "unicamp"          # subcarpeta en env.DATA_ROOT
IMAGES_DIR  = "Unicamp-UAV-Dataset-main/Test Images - Sample"   # relativo a UNICAMP_DIR
LABELS_DIR  = "Unicamp-UAV-Dataset-main/Test Labels - Sample"

OUT_DIR = "MyDrive/deteccion_drones/runs/rgb"
RUN     = "visdrone_person_yolo26s_1024_20260929-1747"

CONFS   = (0.25, 0.35, 0.5)      # umbrales de operacion a comparar
TILE, OVERLAP = 1280, 0.2        # recortes para 4K
LIMIT   = 50                     # None = todo el test

## Setup

In [ ]:
REPO = "https://github.com/sbstn-sss/deteccion_drones.git"
REPO_DIR = "/content/deteccion_drones"
BRANCH = "feat/framework"   # volver a "main" cuando se mergee

import sys
if "google.colab" in sys.modules:
    !git -C {REPO_DIR} checkout -q {BRANCH} 2>/dev/null && git -C {REPO_DIR} pull -q 2>/dev/null || git clone -q -b {BRANCH} {REPO} {REPO_DIR}
    !pip install -q -e {REPO_DIR}
    sys.path.insert(0, f"{REPO_DIR}/src")   # pip -e usa un .pth que el kernel ya iniciado no lee: sin esto falla el import

try:  # el IPython de Colab con Python 3.13 trae un autoreload que importa 'imp' (eliminado en 3.12)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except ModuleNotFoundError:
    print("autoreload no disponible: reinicia el kernel tras editar src/")
import importlib; importlib.invalidate_caches()   # el repo se clono despues de arrancar el kernel
from deteccion import env, person_eval
from deteccion import Experiment
env.mount_drive()

## 1. Obtener el test

Idempotente (marcador `.done`). La ultima linea lista las carpetas de test encontradas: si no coinciden con `IMAGES_DIR` / `LABELS_DIR`, corrige los parametros.

In [ ]:
unicamp_dir = env.DATA_ROOT / UNICAMP_DIR
env.unzip_once(env.drive_path(UNICAMP_ZIP), unicamp_dir)
sorted(str(p.relative_to(unicamp_dir)) for p in unicamp_dir.rglob("*") if p.is_dir() and "test" in p.name.lower())

## 2. Revisar el test

Cuantas imagenes, cuantas con label, resolucion y formato de las labels (se espera YOLO: `clase xc yc w h` normalizados).

In [ ]:
from PIL import Image

images_dir, labels_dir = unicamp_dir / IMAGES_DIR, unicamp_dir / LABELS_DIR
pairs = person_eval.pair_images(images_dir, labels_dir)
print(f"{len(pairs)} imagenes, {pairs.has_label.sum()} con label")
print("resoluciones:", {Image.open(p).size for p in pairs.image_path.head(20)})
print(pairs[pairs.has_label].label_path.iloc[0].read_text().splitlines()[:5])

## 3. Cargar el modelo

Solo `best.pt` del run; no hace falta el dataset de VisDrone. `imgsz` es el del entrenamiento.

In [ ]:
exp = Experiment.load(env.drive_path(OUT_DIR) / RUN)
model, IMGSZ = exp.model, exp.cfg.imgsz
print(RUN, "| imgsz", IMGSZ, "| clases de persona:", [model.names[i] for i in person_eval.person_class_ids(model.names)])

## 4. Frame completo vs recortes

- **Frame completo:** como se despliega hoy. Un frame 4K se achica a `IMGSZ`: las personas quedan varias veces mas chicas.
- **Recortes:** el frame se corta en tiles de `TILE` px con solape y se une con NMS. Mas recall en 4K, pero ~8 inferencias por frame.

La columna que importa es **R** (personas encontradas) al umbral que usarias en operacion, sin que **P** se desplome.

In [ ]:
import pandas as pd

full  = person_eval.evaluate_persons(model, images_dir, labels_dir, CONFS, imgsz=IMGSZ, limit=LIMIT)
tiled = person_eval.evaluate_persons(model, images_dir, labels_dir, CONFS, imgsz=IMGSZ, limit=LIMIT,
                                     tile=TILE, overlap=OVERLAP)
pd.concat({"frame completo": full, f"recortes {TILE}": tiled}).round(3)

## 5. A ojo

Azul = persona real, verde = deteccion del modelo. Mismas imagenes cada vez (`seed`).

In [ ]:
person_eval.show_persons(model, images_dir, labels_dir, n=3, seed=0, conf=0.35, imgsz=IMGSZ)

In [ ]:
person_eval.show_persons(model, images_dir, labels_dir, n=3, seed=0, conf=0.35, imgsz=IMGSZ, tile=TILE, overlap=OVERLAP)

## Conclusiones

_(completar)_
- Recall persona, frame completo vs recortes (conf 0.35):
- Precision: ¿se mantiene con recortes?
- Donde falla a ojo (tamano, oclusion, sombras):
- ¿Vale el costo de los recortes para el outpost?